# Week 11 Demo: Text Data Preparation

We will prepare real SMS messages for classification, combine the text with two simple numeric features, and explain the resulting feature space. Run the notebook from this folder and keep all outputs. Dataset attribution and rules are in DATA_GUIDE.txt. 

## Block 1: Environment and imports

Use your existing course .venv. The course dependency remains scikit-learn 1.9.0. The version printout records the environment that produced these saved outputs.

In [35]:
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

Python: 3.14.4
scikit-learn: 1.9.0
pandas: 3.0.5


## Block 2: The SMS data

A spam filter is a classifier. Each row contains a message and its label. Ham means a legitimate message. The target is 1 for spam. Read DATA_GUIDE.txt.

In [36]:
raw_df = pd.read_csv('data/sms_messages.csv')
print('Raw shape:', raw_df.shape)
print(raw_df.dtypes)
display(raw_df.head())
display(raw_df['is_spam'].value_counts().sort_index().rename(index={0: 'ham', 1: 'spam'}))

Raw shape: (4609, 3)
record_id      str
text           str
is_spam      int64
dtype: object


,record_id,text,is_spam
0,SMS00001,"Go until jurong point, crazy.. Available only ...",0
1,SMS00002,Ok lar... Joking wif u oni...,0
2,SMS00003,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,SMS00004,U dun say so early hor... U c already then say...,0
4,SMS00005,"Nah I don't think he goes to usf, he lives aro...",0


is_spam
ham     4020
spam     589
Name: count, dtype: int64

## Block 3: Examine text quality

Before fitting anything, check the text itself. Missing text (NaN) stops a vectorizer with an error. Blank text, or text that contains no usable tokens, quietly becomes a row of zeros. The default analyzer below uses the same tokenization rules as the vectorizers we fit later, so it shows which messages will contribute no text features at all.

In [37]:
analyzer = CountVectorizer().build_analyzer()

def token_count(text):
    return len(analyzer(text))

text = raw_df['text']
missing_text = text.isna()
blank_text = text.fillna('').str.strip().eq('')
no_tokens = ~missing_text & ~blank_text & text.fillna('').map(token_count).eq(0)
print('Missing text:', int(missing_text.sum()))
print('Blank or whitespace-only text:', int(blank_text.sum()))
print('Text with no usable tokens:', int(no_tokens.sum()))
display(raw_df.loc[no_tokens, ['record_id', 'text', 'is_spam']])
display(text.fillna('').str.len().describe().round(1).to_frame('characters per message'))

Missing text: 0
Blank or whitespace-only text: 0
Text with no usable tokens: 4


,record_id,text,is_spam
2749,SMS03377,:),0
3518,SMS04294,G.W.R,0
3965,SMS04825,:-) :-),0
4268,SMS05176,U 2.,0


,characters per message
count,4609.0
mean,75.4
std,54.9
min,2.0
25%,34.0
50%,55.0
75%,111.0
max,629.0


The messages above are real messages, such as emoticons, so we keep them and record their IDs. Their text features will be all zeros, but the numeric features we add in Block 5 still describe them. A different task could reasonably decide to route such messages to a person instead.

## Block 4: Validity and duplicate checks

Repeated messages could otherwise land in both training and test sets. We compare lowercased, stripped text before splitting, check for conflicting labels, and keep the original wording for modeling. Exact-equivalent duplicate checks do not remove every related spam template.

In [38]:
working_df = raw_df.copy()
invalid = (working_df['text'].isna()
           | working_df['text'].str.strip().eq('')
           | ~working_df['is_spam'].isin([0, 1]))
rejected_ids = working_df.loc[invalid, 'record_id'].tolist()
working_df = working_df.loc[~invalid].copy()
working_df['duplicate_key'] = working_df['text'].str.strip().str.lower()
conflicts = working_df.groupby('duplicate_key')['is_spam'].nunique()
assert conflicts.le(1).all(), 'Investigate conflicting duplicate labels before proceeding.'
duplicates = working_df.duplicated('duplicate_key', keep='first')
duplicate_ids = working_df.loc[duplicates, 'record_id'].tolist()
working_df = working_df.loc[~duplicates].copy()
print('Rejected invalid rows:', len(rejected_ids))
print('Removed duplicate rows:', len(duplicate_ids))
print('Retained rows:', len(working_df))
assert len(raw_df) == len(rejected_ids) + len(duplicate_ids) + len(working_df)
assert working_df['duplicate_key'].is_unique

Rejected invalid rows: 0
Removed duplicate rows: 364
Retained rows: 4245


## Block 5: Add two numeric features

Text often arrives alongside structured columns. Our SMS file has none, so we derive two simple numeric features from each message, using the definitions in DATA_GUIDE.txt: the number of characters (after removing surrounding spaces) and the number of digits. Each value is calculated from its own row only, so nothing is learned from other rows and it is safe to calculate before the split. Anything that *learns* from data, such as the vocabulary, IDF weights, or scaler means, stays inside the pipeline. The same function must be applied to every new message.

In [39]:
def add_message_features(df):
    """Return a copy with char_count and digit_count calculated from each row's text."""
    out = df.copy()
    text = out['text'].fillna('').str.strip()
    out['char_count'] = text.str.len()
    out['digit_count'] = text.str.count(r'\d')
    return out

working_df = add_message_features(working_df)
display(working_df.groupby('is_spam')[['char_count', 'digit_count']].median()
        .rename(index={0: 'ham', 1: 'spam'}))

,char_count,digit_count
is_spam,,
ham,49.0,0.0
spam,148.0,16.0


## Block 6: Training and test rows

Stratification approximately preserves class proportions. The predictors are now a DataFrame with one text column and two numeric columns. We reserve test rows for a final evaluation of this fixed workflow. We do not use test scores to choose cleaning rules or model settings.

In [40]:
TEXT_COLUMN = 'text'
NUMERIC_COLUMNS = ['char_count', 'digit_count']
FEATURES = [TEXT_COLUMN] + NUMERIC_COLUMNS

train_df, test_df = train_test_split(
    working_df, test_size=0.20, random_state=42, stratify=working_df['is_spam'])
X_train, X_test = train_df[FEATURES], test_df[FEATURES]
y_train, y_test = train_df['is_spam'], test_df['is_spam']
assert set(train_df['duplicate_key']).isdisjoint(test_df['duplicate_key'])
print('Training rows:', len(X_train), 'Test rows:', len(X_test))
display(pd.DataFrame({'train_share': y_train.value_counts(normalize=True),
                      'test_share': y_test.value_counts(normalize=True)}).sort_index())

Training rows: 3396 Test rows: 849


,train_share,test_share
is_spam,,
0,0.882214,0.882214
1,0.117786,0.117786


## Block 7: What tokenization keeps

A token is a unit extracted from text. These vectorizers lowercase by default and keep word-like tokens of at least two characters. Punctuation separates tokens. Inspect what happens to NOT and the single-letter I. Removing stop words can erase useful meaning. Bigrams include adjacent pairs, such as not happy.

In [41]:
example = "I am NOT happy with this phone!"
print('Default tokens:', CountVectorizer().build_analyzer()(example))
print('With English stop words:', CountVectorizer(stop_words='english').build_analyzer()(example))
print('With unigrams and bigrams:', CountVectorizer(ngram_range=(1, 2)).build_analyzer()(example))

Default tokens: ['am', 'not', 'happy', 'with', 'this', 'phone']
With English stop words: ['happy', 'phone']
With unigrams and bigrams: ['am', 'not', 'happy', 'with', 'this', 'phone', 'am not', 'not happy', 'happy with', 'with this', 'this phone']


## Block 8: Counts and TF-IDF

A document-term matrix has one row per document and one column per learned term. Counts record occurrences. TF-IDF reweights counts using how widely a term appears in the training documents, then normalizes each nonzero row by default. It does not measure truth or understand meaning. We convert only this tiny teaching matrix to a dense array.

In [42]:
toy_text = ['free phone phone', 'call phone', 'free prize']
counts = CountVectorizer()
count_matrix = counts.fit_transform(toy_text)
display(pd.DataFrame(count_matrix.toarray(), columns=counts.get_feature_names_out()))
tfidf_toy = TfidfVectorizer()
tfidf_matrix = tfidf_toy.fit_transform(toy_text)
display(pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_toy.get_feature_names_out()).round(3))
# These three invented documents only explain the representation.
# They are separate from the SMS model and its train/test split.

,call,free,phone,prize
0,0,1,2,0
1,1,0,1,0
2,0,1,0,1


,call,free,phone,prize
0,0.000,0.447,0.894,0.000
1,0.796,0.000,0.605,0.000
2,0.000,0.605,0.000,0.796


## Block 9: One workflow for text and numbers

A ColumnTransformer routes each column to the preparation it needs, as in Weeks 4 to 6:

* **text branch:** fill any missing text with an empty string, then apply TfidfVectorizer. The vectorizer needs one document per row, so the column is named with a plain string, `'text'`, not a list.
* **numeric branch:** StandardScaler on the two numeric columns.

Logistic regression is familiar from Week 9. `min_df=2` keeps terms found in at least two training documents. We keep stop words and use individual words for this fixed model.

In [43]:
def fill_missing_text(text):
    """Replace missing documents with empty strings so the vectorizer can read every row."""
    return pd.Series(text).fillna('')


text_prep = Pipeline([
    ('fill_missing', FunctionTransformer(fill_missing_text, feature_names_out='one-to-one')),
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words=None, ngram_range=(1, 1), min_df=2)),
])
preprocessor = ColumnTransformer([
    ('text', text_prep, TEXT_COLUMN),            # a string: one document per row
    ('numeric', StandardScaler(), NUMERIC_COLUMNS),  # a list: a 2-D table of numbers
])
model = Pipeline([
    ('prep', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
print('Pipeline fitted using training rows only.')

Pipeline fitted using training rows only.


## Block 10: The learned feature space

The ColumnTransformer places the text columns and the two scaled numeric columns side by side in one sparse matrix, which stores nonzero values efficiently. Training and test sets have the same columns because transform reuses the training vocabulary, IDF, and scaler values. Keep the real matrices sparse.

In [44]:
fitted_prep = model.named_steps['prep']
fitted_tfidf = fitted_prep.named_transformers_['text'].named_steps['tfidf']
train_features = fitted_prep.transform(X_train)
test_features = fitted_prep.transform(X_test)
feature_names = fitted_prep.get_feature_names_out()
text_feature_count = sum(name.startswith('text__') for name in feature_names)
print('Training feature shape:', train_features.shape)
print('Test feature shape:', test_features.shape)
print('Text features:', text_feature_count, ' Numeric features:', len(feature_names) - text_feature_count)
print('Output type:', type(train_features).__name__)
print('Stored nonzero training values:', train_features.nnz)
print('Training density:', round(train_features.nnz / (train_features.shape[0] * train_features.shape[1]), 4))
print('First 5 names:', feature_names[:5].tolist())
print('Last 2 names:', feature_names[-2:].tolist())
recognized_terms = fitted_tfidf.transform(fill_missing_text(X_train[TEXT_COLUMN])).getnnz(axis=1)
print('Training messages with no recognized terms:', int((recognized_terms == 0).sum()))
assert train_features.shape[1] == test_features.shape[1] == len(feature_names)

Training feature shape: (3396, 2721)
Test feature shape: (849, 2721)
Text features: 2719  Numeric features: 2
Output type: csr_matrix
Stored nonzero training values: 45174
Training density: 0.0049
First 5 names: ['text__00', 'text__000', 'text__02', 'text__03', 'text__04']
Last 2 names: ['numeric__char_count', 'numeric__digit_count']
Training messages with no recognized terms: 10


## Block 11: A common mistake: a list instead of a column name

Data School (Ch. 6.5) warns about this one. `['text']` hands the vectorizer a one-column *table*. A vectorizer iterates over what it receives, and iterating over a table gives its column names, so the whole training set becomes a single "document" containing the word text. On its own, this produces no error. Always check the shape.

In [45]:
wrong = ColumnTransformer([('text', TfidfVectorizer(), ['text'])])
right = ColumnTransformer([('text', TfidfVectorizer(), 'text')])
print('With a list  ["text"]:', wrong.fit_transform(X_train).shape)
print('With a string "text" :', right.fit_transform(X_train).shape)
# Both use default settings (no min_df), so the correct version keeps every training term.
# These two transformers are only for this check. Our model above is unchanged.

With a list  ["text"]: (1, 1)
With a string "text" : (3396, 6496)


## Block 12: Model and baseline evaluation

The baseline learns the most frequent training label and predicts it for every test row. It ignores the features. We evaluate both on identical test rows. Precision and recall treat spam as the positive class. Accuracy alone can hide missed spam.

In [46]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
pred = model.predict(X_test)

def scores(y_true, y_pred):
    return [accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, zero_division=0),
            recall_score(y_true, y_pred, zero_division=0)]

results = pd.DataFrame({'model': scores(y_test, pred),
                        'majority_baseline': scores(y_test, baseline_pred)},
                       index=['accuracy', 'precision (spam)', 'recall (spam)'])
display(results.round(3))
print('A baseline that never predicts spam has no spam predictions to be precise about; zero_division=0 reports that as 0.')
cm = confusion_matrix(y_test, pred, labels=[0, 1])
display(pd.DataFrame(cm, index=['actual ham', 'actual spam'], columns=['predicted ham', 'predicted spam']))

,model,majority_baseline
accuracy,0.98,0.882
precision (spam),1.00,0.000
recall (spam),0.83,0.000


A baseline that never predicts spam has no spam predictions to be precise about; zero_division=0 reports that as 0.


,predicted ham,predicted spam
actual ham,749,0
actual spam,17,83


## Block 13: Evidence from mistakes

A false positive marks legitimate text as spam. A false negative misses spam. Read individual mistakes alongside the confusion matrix. Describe what the wording suggests, while recognizing that one example cannot prove the cause of an error. We do not refit after inspecting these test errors.

In [47]:
spam_column = list(model.classes_).index(1)
review = test_df[['record_id', 'text', 'char_count', 'digit_count', 'is_spam']].copy()
review['predicted'] = pred
review['spam_probability'] = model.predict_proba(X_test)[:, spam_column].round(3)
mistakes = review.loc[review['is_spam'] != review['predicted']]
print('False positives:', int(cm[0, 1]), 'False negatives:', int(cm[1, 0]))
with pd.option_context('display.max_colwidth', None):
    display(mistakes.head(5))

False positives: 0 False negatives: 17


,record_id,text,char_count,digit_count,is_spam,predicted,spam_probability
29,SMS00035,Thanks for your subscription to Ringtone UK your mobile will be charged £5/month Please confirm by replying YES or NO. If you reply NO you will not be charged,158,1,1,0,0.254
2877,SMS03531,"Xmas & New Years Eve tickets are now on sale from the club, during the day from 10am till 8pm, and on Thurs, Fri & Sat night this week. They're selling fast!",157,3,1,0,0.265
2374,SMS02916,Sorry! U can not unsubscribe yet. THE MOB offer package has a min term of 54 weeks> pls resubmit request after expiry. Reply THEMOB HELP 4 more info,148,3,1,0,0.179
58,SMS00069,"Did you hear about the new ""Divorce Barbie""? It comes with all of Ken's stuff!",78,0,1,0,0.023
910,SMS01106,Message Important information for O2 user. Today is your lucky day! 2 find out why log onto http://www.urawinner.com there is a fantastic surprise awaiting you,159,2,1,0,0.424


### Borderline cases

The classifier labels a message as spam when its predicted probability is above 0.5, the default **decision threshold**. Predictions close to that threshold are called **borderline cases** (or **uncertain predictions**). A small change in the wording or the numeric features could flip their label. Error analysis looks at them alongside the mistakes, because they show what pushes the model toward one class even when it ends up right. The table lists the five test predictions closest to the threshold.

In [48]:
distance = np.abs(model.predict_proba(X_test)[:, spam_column] - 0.5)
borderline = review.assign(distance_from_threshold=distance.round(3)).sort_values('distance_from_threshold')
with pd.option_context('display.max_colwidth', None):
    display(borderline.head(5))

,record_id,text,char_count,digit_count,is_spam,predicted,spam_probability,distance_from_threshold
3312,SMS04048,Win a £1000 cash prize or a prize worth £5000,45,8,1,1,0.512,0.012
3480,SMS04249,Text PASS to 69669 to collect your polyphonic ringtones. Normal gprs charges apply only. Enjoy your tones,105,5,1,1,0.521,0.021
1412,SMS01725,"Hi Jon, Pete here, Ive bin 2 Spain recently & hav sum dinero left, Bill said u or ur rents mayb interested in it, I hav 12,000pes, so around £48, tb, James.",157,8,0,0,0.478,0.022
1061,SMS01291,"Hey...Great deal...Farm tour 9am to 5pm $95/pax, $50 deposit by 16 May",70,8,0,0,0.442,0.058
2498,SMS03068,Get a brand new mobile phone by being an agent of The Mob! Plus loads more goodies! For more info just text MAT to 87021.,121,5,1,1,0.560,0.060


**What practitioners do with borderline cases.** 

1. **Human review (human-in-the-loop).** Instead of acting automatically, a deployed filter can send predictions within a set distance of the threshold to a review queue or a junk folder, where a person decides.
2. **Threshold selection.** If false positives cost more than false negatives, a team might raise the threshold, and lower it in the opposite case. The new threshold must be chosen on a **validation set**, never on the test set. Choosing it from the rows above would turn the test set into a tuning set (Week 10).
3. **Labeling more data (active learning).** Borderline examples are the most informative ones to label next. Uncertainty sampling sends them to people for labels, then the model is retrained on the larger training set.
4. **Feature review.** Patterns in borderline cases, such as many digits in legitimate messages, can suggest data or feature changes for a future version. Those changes would be evaluated on new data, not on this test set.

## Block 14: New messages, missing text, and unknown words

New rows go through the same steps: derive the numeric features with the same function, then pass the DataFrame to the fitted pipeline. The vocabulary does not change. These five rows are invented. One has missing text, one is only spaces, and one contains only an unknown word. First, see what the fitted vectorizer does if it receives the raw text column directly, without the fill step.

In [49]:
new_df = pd.DataFrame({
    'record_id': ['NEW1', 'NEW2', 'NEW3', 'NEW4', 'NEW5'],
    'text': ['Can we meet after class tomorrow?',
             'Claim your free prize now. Call 09000000000 today!',
             None, '   ', 'zzzxxyyqq']})

try:
    fitted_tfidf.transform(new_df['text'])
except ValueError as err:
    print('Vectorizer without the fill step:', err)

Vectorizer without the fill step: np.nan is an invalid document, expected byte or unicode string.


In [50]:
vocab_before = fitted_tfidf.get_feature_names_out().copy()
new_ready = add_message_features(new_df)
new_result = new_ready[['record_id', 'text', 'char_count', 'digit_count']].copy()
new_result['recognized_terms'] = fitted_tfidf.transform(fill_missing_text(new_ready['text'])).getnnz(axis=1)
new_result['predicted_is_spam'] = model.predict(new_ready[FEATURES])
new_result['spam_probability'] = model.predict_proba(new_ready[FEATURES])[:, spam_column].round(3)
display(new_result)
assert np.array_equal(vocab_before, fitted_tfidf.get_feature_names_out())
print('Vocabulary unchanged after prediction:', True)

,record_id,text,char_count,digit_count,recognized_terms,predicted_is_spam,spam_probability
0,NEW1,Can we meet after class tomorrow?,33,0,6,0,0.005
1,NEW2,Claim your free prize now. Call 09000000000 to...,50,11,7,1,0.932
2,NEW3,NaN,0,0,0,0,0.009
3,NEW4,,0,0,0,0,0.009
4,NEW5,zzzxxyyqq,9,0,0,0,0.009


Vocabulary unchanged after prediction: True


Rows with missing, blank, or unknown-only text have zero recognized terms. The pipeline still returns a prediction from the numeric features and the intercept. That output is not evidence that the model understood the message. A production workflow might flag such rows for review instead of acting on them automatically.

## Block 15: Answers and interpretation

The printed answers summarize the observed results. Consider the cost of blocking a legitimate message as well as missing spam. Historical classroom evaluation is only one piece of evidence.

In [51]:
print(f"Model accuracy: {results.loc['accuracy', 'model']:.3f}")
print(f"Baseline accuracy: {results.loc['accuracy', 'majority_baseline']:.3f}")
print(f"Spam precision: {results.loc['precision (spam)', 'model']:.3f}  Spam recall: {results.loc['recall (spam)', 'model']:.3f}")

Model accuracy: 0.980
Baseline accuracy: 0.882
Spam precision: 1.000  Spam recall: 0.830


### Key questions

- **Why split before fitting?** Vocabulary, IDF, and scaler values must learn only from training rows.
- **Why a `ColumnTransformer`?** Text and numeric columns need different preparation in one reusable workflow.
- **Why `'text'` and not `['text']`?** A vectorizer needs one document per row.
- **Why fill missing text inside the pipeline?** New rows can arrive with missing text even when training rows had none.
- **Why keep stop words?** Words such as *not* can change meaning.

These historical results do not establish performance on future messages. Historical classroom evaluation is only one piece of evidence.